# Matrix factorization with focal loss: positive vs negative, with negative sampling

Logistic matrix factorization of the **user × recipe** matrix, trained with a class-weighted **focal loss** and fitted by
**alternating Newton steps** (ALS-style: users, then recipes), as a **binary** task: does the user like the recipe?

**Data** (`input_stage2/recipe_user_ratings.csv`)

- **Preprocessing** (first section): every row is one rating, *who* (`user_id`) rated *which* recipe. The recipe's product /
  adj / verb tokens are normalized, lemmatized, singularized and lowercased, and its nutrients are standardized, once per recipe.
  A plot shows the positive and negative rating points.
- Every rating 0–5 is kept (0 is a review without stars).
- **Labels:** a rating of 4–5★ is **positive** (1); a rating of 0–3★ is **negative** (0).
- If a user rated the same recipe more than once, only the latest rating is kept.
- Only users with at least 6 ratings are kept. Every row of the other users is discarded.
- **Split**, per user by date: the latest rating is **test**, the one before it is **valid**, and the rest (at least 4) are **train**.
  Valid picks the regularization strengths and the iteration to keep. Test is used only for the final report.
- **Negative sampling:** every positive rating is paired with 1 recipe the user never rated (1 positive : 1 negative). They are labelled
  0 (negative) and go to the positive's split. See the negative sampling section.
- **Two ways to handle the imbalance**, each trained, evaluated and saved on its own: **negative sampling** (add unobserved recipes
  as negatives; saved to `model_artifacts/mf_als/`) and **undersampling with One-Sided Selection** (the last section: observed
  ratings only, redundant and borderline positive train rows removed; saved to `model_artifacts/mf_als_one_sided_selection/`).

**Model and loss**

$$\eta_{ui} = \mu + b_u + c_i + p_u^\top q_i, \qquad \hat p_{ui} = \sigma(\eta_{ui})$$

$\eta_{ui}$ is the logit and $\hat p_{ui}$ the probability of a positive. $\mu$ is the logit of the weighted train positive rate, $b_u$ and $c_i$
are the user and recipe biases, and $p_u, q_i \in \mathbb{R}^k$ are the latent factors. Training minimizes a class-weighted **focal loss**
plus an **L2 penalty**:

$$L = \sum_{(u,i) \in \text{train}} w_{y_{ui}}\, \underbrace{\big(1 - p_t\big)^{\gamma}}_{\text{modulating factor}} \big(-\log p_t\big) + \lambda \Big(\sum_u \lVert p_u \rVert^2 + \sum_i \lVert q_i \rVert^2\Big) + \lambda_b \Big(\sum_u b_u^2 + \sum_i c_i^2\Big),
\qquad p_t = \begin{cases} \hat p_{ui} & y_{ui} = 1 \\ 1 - \hat p_{ui} & y_{ui} = 0 \end{cases}$$

$-\log p_t$ is the binary cross-entropy (BCE). The **modulating factor** $(1 - p_t)^{\gamma}$ ($\gamma$ = `FOCAL_GAMMA`) scales it down for rows the
model already gets right ($p_t$ near 1), so training concentrates on the hard ones. $\gamma = 0$ is plain BCE.

**Class weights** ($\alpha$ in the focal loss). With 1 sampled negative per positive plus the real 0–3★ ratings, train is only slightly more negative than positive (about 52 : 48).
A row of class $c$ gets the weight $w_c \propto \big(n / (2\, n_c)\big)^{\beta}$, scaled so that the mean train weight is 1. Here $n_c$ is the number of
train rows of class $c$, $n$ is the number of all train rows, and $\beta$ = `WEIGHT_BALANCE`. $\beta = 0$ is unweighted. $\beta = 1$ is scikit-learn's
`"balanced"`: both classes carry the same total weight. Values in between balance partly.

**Fitting: alternating Newton steps.** Squared error has a closed-form ALS step; the focal loss does not. With every recipe's $(q_i, c_i)$
fixed, $L$ still splits into one small problem per user, and a **Newton step** on it is a weighted ridge regression (iteratively
reweighted least squares). With $g$ and $h$ the first and second derivatives of the focal loss with respect to $\eta$ at the current model,
the quadratic approximation of $L$ is minimized by

$$\begin{bmatrix} p_u \\ b_u \end{bmatrix} = \big(Z_u^\top \Omega_u Z_u + \Lambda\big)^{-1} Z_u^\top \Omega_u\, t_u, \qquad t = \eta - \mu - c_i - g / h, \qquad \Omega = \operatorname{diag}\!\big(w\, h / 2\big)$$

with $Z_u = \begin{bmatrix} q_i^\top & 1 \end{bmatrix}_{i \in I_u}$ and $\Lambda = \operatorname{diag}(\lambda, \dots, \lambda, \lambda_b)$, the same batched solve as least-squares ALS.
The recipe step is the same with users and recipes swapped. The focal loss is not convex for $\gamma > 0$ ($h$ turns negative on badly
misclassified rows), so $h$ is floored at `MIN_CURVATURE`, and each step is followed by a **backtracking line search** that keeps only a
step size that does not raise $L$. So $L$ never increases.

**Why the penalty matters.** Without it, a recipe with a single train row could fit that label exactly (its logit would run off to
$\pm\infty$). With it, parameters backed by little weight stay close to 0. A small $\lambda$ gives a low train loss but a high valid loss
(overfitting). A very large $\lambda$ shrinks every factor to 0, which leaves the bias baseline.

**Metrics.** A probability of at least `DECISION_THRESHOLD` (0.5) is predicted **positive**. For each class (negative and positive):
**precision**, **recall** and **F1**; also accuracy and macro F1, and from the probability **ROC AUC** and **average precision**.
The probability itself is scored by the **log loss** (BCE) and a **weighted log loss**: the mean of the per-class log losses, so both
classes count the same however rare. The weighted log loss depends on neither `WEIGHT_BALANCE` nor `FOCAL_GAMMA`, so runs with
different class weights or $\gamma$ stay comparable.

**Baselines.** Two simpler models show what the factors add:
- **base rate**: one constant probability $\sigma(\mu)$ for every row, the "predict the prior" baseline (like scikit-learn's
  `DummyClassifier`). With balanced class weights $\sigma(\mu) = 0.5$, so it predicts every row positive and has ROC AUC 0.5.
- **user + recipe bias**: $\sigma(\mu + b_u + c_i)$, the model without factors ($k = 0$). It knows how generous each user is and how
  well liked each recipe is, but ranks recipes the same way for every user (no personalization).

**Tuning.** $\lambda_b$ is picked first on the **bias baseline** ($k = 0$).
Then $\lambda$ is picked for the $k$-factor model with that $\lambda_b$. Both choices, and early stopping, use the valid metric `SELECT_BY`
(weighted log loss by default). A recipe with no train row gets $q_i = 0$ and $c_i = 0$, so its logit falls back to $\mu + b_u$.


In [31]:
# Run once if these packages are missing, then restart the notebook kernel.
# %pip install numpy pandas scipy scikit-learn matplotlib
import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.special import expit, log_expit, logit
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score)

In [32]:
RANDOM_STATE = 42
NOTEBOOK_DIR = next((p for p in (Path.cwd(), Path.cwd() / "DataMappingRecommendation")
                     if (p / "input_stage2/recipe_user_ratings.csv").is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Run from DataMappingRecommendation or its project root.")
DATA_PATH = NOTEBOOK_DIR / "input_stage2/recipe_user_ratings.csv"
ALS_DIR = NOTEBOOK_DIR / "model_artifacts/mf_als"

POSITIVE_MIN = 4  # Ratings 4-5 are positive (label 1), ratings 0-3 negative (label 0). Every rating 0-5 is kept (0 = a review without stars).
LABELS = [0, 1]  # negative, positive. Scores are clipped to [0, 1].
MIN_USER_RATINGS = 6  # Users with fewer ratings are discarded, with all their rows.
NEGATIVES_PER_POSITIVE = 1  # Each positive rating is paired with this many recipes the user never rated, labelled 0 (1 : 1).
DECISION_THRESHOLD = 0.5  # A clipped score at or above this is predicted positive.
WEIGHT_BALANCE = 1.0  # beta: class weight (alpha) (n / (2 n_c)) ** beta. 0 = unweighted, 1 = "balanced" (both classes weigh the same).
FOCAL_GAMMA = 2.0  # gamma: the focal loss modulating factor (1 - p_t) ** gamma. 0 = plain binary cross-entropy.
SELECT_BY = "weighted log loss"  # The valid metric that picks lambda_b, lambda, and the kept iteration: "weighted log loss" or "log loss".

FACTORS = 32  # k: the size of p_u and q_i.
L2_BIAS_GRID = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]  # lambda_b candidates, tried on the bias baseline (k = 0).
# The focal loss is ~(1 - p_t)^gamma times smaller than BCE (1/4 at p_t = 0.5 with gamma = 2), so the same lambda penalizes
# relatively more than with BCE or squared error: the grids reach lower values.
L2_GRID = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]  # lambda candidates for the factors.
INIT_STD = 0.1  # q_i starts as N(0, INIT_STD^2); the first user step solves the user side from it.
MIN_CURVATURE = 1e-6  # Floor on the focal loss's second derivative h (it turns negative for badly misclassified rows).
LINE_SEARCH_STEPS = (1.0, 0.5, 0.25, 0.125, 0.0625)  # Newton step sizes tried in order; the first that does not raise L is kept.
MAX_ITERATIONS = 40  # One iteration is a user step followed by a recipe step (Newton steps from small factors start slowly).
EARLY_STOPPING_PATIENCE = 3  # Stop after this many iterations without a better valid SELECT_BY,
MIN_IMPROVEMENT = 1e-4  # where better means lower by more than this.
CHUNK_ELEMENTS = 2**22  # At most this many float64 values (32 MB) per array in a batch of normal equations.
LOGIT_CHUNK = 2**19  # Rows per chunk when computing logits (bounds the gathered factor arrays).
print(f"Ratings: {DATA_PATH}\nArtifacts: {ALS_DIR}")

Ratings: /Users/tung/Desktop/Học/Projects/ProjectSem1_2/DataMappingRecommendation/input_stage2/recipe_user_ratings.csv
Artifacts: /Users/tung/Desktop/Học/Projects/ProjectSem1_2/DataMappingRecommendation/model_artifacts/mf_als


## Data preprocessing: who rated what, recipe tokens and nutrients

**Every row is one rating:** user `user_id` rated recipe `recipe_id` with `rating` on `date`. So a recipe appears once for every user
who rated it: 1,132,367 rating rows cover 231,637 recipes and 226,570 users. A recipe's content (its product / adj / verb tokens and
its nutrients) is the same on all of its rows, so it is cleaned and standardized **once per recipe**. Standardizing over the rating
rows instead would let the recipes with many raters dominate the mean and standard deviation.

**Tokens.** `product`, `adj` and `verb` hold one list of spans per ingredient, e.g. `[["winter squash"], ["olive oil"], ...]`. Every span is
1. **normalized**: accents stripped to ASCII (`jalapeño` → `jalapeno`), lowercased, bracketed text and punctuation removed, stopwords dropped;
2. **lemmatized** with WordNet by part of speech: nouns for `product`, adjectives for `adj`, verbs for `verb` (`chopped` → `chop`, `frozen` → `freeze`);
3. **singularized** (`tomatoes` → `tomato`, `leaves` → `leaf`, while `molasses` and `hummus` stay as they are);
4. **lowercased** again, and repeated spans of a recipe are kept once.

These are the rules of `normalize_entity_vocab` in `map_recipe_nutrients.ipynb`, copied here, plus the accent stripping.

**Nutrients.** The seven `(g)` columns sit on very different and very skewed scales: this file stores Food.com values multiplied by the
daily value (calories × 2000), so calories reach the hundreds of millions. Each column gets `log1p` (to tame the right tail) and is then
**standardized** to mean 0 and standard deviation 1 over the recipes. A z-score does not depend on a column's unit or scale factor.

The matrix factorization below uses only user, recipe, rating and date; these cleaned features are for inspecting the data and for the plot.


In [ ]:
import re
import unicodedata
from functools import lru_cache

from nltk.stem import WordNetLemmatizer

# ---------- the rules of normalize_entity_vocab in map_recipe_nutrients.ipynb ----------
IRREGULAR_PLURALS = {
    # classic irregulars
    "geese": "goose", "mice": "mouse", "lice": "louse", "teeth": "tooth",
    "feet": "foot", "children": "child", "men": "man", "women": "woman",
    "people": "person", "oxen": "ox", "testes": "testis",
    # -f / -fe stems: no rule separates "loaves" from "olives"
    "loaves": "loaf", "leaves": "leaf", "halves": "half", "knives": "knife",
    "calves": "calf", "hooves": "hoof", "shelves": "shelf", "sheaves": "sheaf",
    "lives": "life", "wolves": "wolf", "thieves": "thief", "scarves": "scarf",
    # the singular already ends in -ie / -i, so -ies -> -y would mangle it
    "pies": "pie", "cookies": "cookie", "brownies": "brownie",
    "smoothies": "smoothie", "veggies": "veggie", "hoagies": "hoagie",
    "toasties": "toastie", "smokies": "smokie", "crunchies": "crunchie",
    "calories": "calorie", "pierogies": "pierogi",
    "chilies": "chili", "chillies": "chilli",
    # the singular already ends in -oe / -e, so -oes -> -o would mangle it
    "toes": "toe", "roes": "roe", "aloes": "aloe", "shoes": "shoe",
    "canoes": "canoe", "mousses": "mousse",
    # a bare -s stem, so the -es rule would leave a trailing -e
    "buses": "bus", "gases": "gas", "octopuses": "octopus",
    # latinate and spanish plurals
    "cacti": "cactus", "fungi": "fungus", "octopi": "octopus",
    "larvae": "larva", "pasteles": "pastel", "nopales": "nopal",
}

# Ends in -s, but is already singular or has no singular at all.
UNCOUNTABLE = {
    # -s words that are not plurals
    "molasses", "series", "species", "news", "haggis", "pancreas", "gras",
    "always", "tennis", "cordyceps", "militaris",
    # mass nouns kept in the plural
    "oats", "grits", "greens", "goods", "brussels", "gyros", "tapas",
    # brands and place names that carry an -s of their own
    "mars", "texas", "snickers", "skittles", "twizzlers", "kudos", "krispies",
    # added here: WordNet or the -s rule would turn these into "mr", "la" and "tran"
    "mrs", "las", "trans",
}

# The regular endings, in order; the first rule that matches wins.
PLURAL_RULES = [
    (re.compile(r"([^aeiou])ies$", re.I), r"\1y"),     # berries -> berry
    (re.compile(r"(ss|sh|ch|x|z)es$", re.I), r"\1"),   # peaches -> peach, boxes -> box
    (re.compile(r"([^aeiou])oes$", re.I), r"\1o"),     # tomatoes -> tomato
    (re.compile(r"([^s])s$", re.I), r"\1"),            # apples -> apple, cheeses -> cheese
]

STOP_WORDS = set(['about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", "he's", 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', 'just', 'll', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'now', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 'same', 'shan', "shan't", 'she', "she'd", "she'll", "she's", 'should', "should've", 'shouldn', "shouldn't", 'so', 'some', 'such', 'than', 'that', "that'll", 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', "they'd", "they'll", "they're", "they've", 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 've', 'very', 'was', 'wasn', "wasn't", 'we', "we'd", "we'll", "we're", "we've", 'were', 'weren', "weren't", 'what', 'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'won', "won't", 'wouldn', "wouldn't", 'you', "you'd", "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves'])

TOKEN_FIELDS = {"product": "n", "adj": "a", "verb": "v"}  # column -> WordNet part of speech
lemmatizer = WordNetLemmatizer()
assert lemmatizer.lemmatize("waffles", "n") == "waffle" and lemmatizer.lemmatize("frozen", "v") == "freeze"  # WordNet is installed


def singularize_word(word):
    low = word.lower()
    if low in IRREGULAR_PLURALS:
        return IRREGULAR_PLURALS[low]
    # no food plural is shorter than four letters; -us is a singular ending ("hummus", "asparagus")
    if low in UNCOUNTABLE or len(low) < 4 or not low.endswith("s") or low.endswith("us"):
        return low
    for rule, replacement in PLURAL_RULES:
        if rule.search(low):
            return rule.sub(replacement, low)
    return low


def normalize_text(text):
    """Accents to ASCII, lowercase, no bracketed text, punctuation or stopwords."""
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode().lower()
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"[^a-z0-9%\s]", " ", text)
    return " ".join(word for word in text.split() if word not in STOP_WORDS and word.strip("%"))


def lemmatize_word(word, pos):
    """WordNet lemma by part of speech, with two guards:
    - words the singular rules keep as they are ("greens", "oats", "snickers") are left alone, since WordNet would strip
      their -s before the rules could protect them;
    - a noun only inflects for the plural, so it is lemmatized only when it looks plural (4+ letters, ends in -s), the
      singular rules' own test. Otherwise WordNet maps names to unrelated words ("roma" -> "rom", "acini" -> "acinus")."""
    if word in UNCOUNTABLE or word in IRREGULAR_PLURALS:
        return word
    if pos == "n" and (len(word) < 4 or not word.endswith("s")):
        return word
    return lemmatizer.lemmatize(word, pos)


@lru_cache(maxsize=None)
def clean_span(span, pos):
    """normalize -> lemmatize (by part of speech) -> singularize -> lowercase, one span at a time (cached)."""
    words = [singularize_word(lemmatize_word(word, pos)).lower() for word in normalize_text(span).split()]
    return " ".join(word for word in words if word)


def clean_token_cell(cell, field):
    """One product / adj / verb cell (a JSON list of span lists, one per ingredient) -> the recipe's cleaned spans, each once."""
    spans = [span for ingredient in json.loads(cell) for span in ingredient]
    cleaned = []
    for span in spans:
        key = clean_span(span, TOKEN_FIELDS[field])
        if key and key not in cleaned:
            cleaned.append(key)
    return cleaned


for field, example in (("product", "Tomatoes"), ("product", "Jalapeño Peppers"), ("product", "turnip greens"),
                       ("product", "instant oats"), ("adj", "fresher"), ("verb", "Deep Fried")):
    print(f"{field:8} {example!r:20} -> {clean_span(example, TOKEN_FIELDS[field])!r}")


In [ ]:
NUTRIENT_COLUMNS = ["calories (g)", "total fat (g)", "sugar (g)", "sodium (g)", "protein (g)", "saturated fat (g)",
                    "carbohydrates (g)"]

rating_rows = pd.read_csv(DATA_PATH, usecols=["user_id", "recipe_id", "rating", "product", "adj", "verb", *NUTRIENT_COLUMNS])
print(f"{len(rating_rows):,} rating rows = {rating_rows['recipe_id'].nunique():,} recipes rated by "
      f"{rating_rows['user_id'].nunique():,} users (a recipe repeats once per user who rated it)")
recipe_groups = rating_rows.groupby("recipe_id")
assert (recipe_groups[["product", "adj", "verb", *NUTRIENT_COLUMNS]].nunique() <= 1).all().all()  # content is per recipe

# One row per recipe: clean its tokens once.
recipe_table = rating_rows.drop_duplicates("recipe_id")[["recipe_id", "product", "adj", "verb", *NUTRIENT_COLUMNS]]
recipe_table = recipe_table.set_index("recipe_id")
rating_rows = rating_rows[["user_id", "recipe_id", "rating"]].copy()  # the per-rating table keeps who, what and how
recipe_groups = rating_rows.groupby("recipe_id")
started = time.time()
for field in TOKEN_FIELDS:
    recipe_table[f"{field}_tokens"] = recipe_table[field].map(lambda cell, field=field: clean_token_cell(cell, field))
print(f"Tokens cleaned in {time.time() - started:.0f}s. Distinct cleaned spans: "
      + ", ".join(f"{field} {len({t for ts in recipe_table[f'{field}_tokens'] for t in ts}):,} "
                  f"(raw {len({s for cell in recipe_table[field] for ing in json.loads(cell) for s in ing}):,})"
                  for field in TOKEN_FIELDS))
display(recipe_table[["product", "product_tokens", "adj_tokens", "verb_tokens"]].head(5))

# Nutrients: log1p, then standardize over the recipes (each recipe once).
nutrient_scaler = StandardScaler().fit(np.log1p(recipe_table[NUTRIENT_COLUMNS].clip(lower=0)))
standardized = nutrient_scaler.transform(np.log1p(recipe_table[NUTRIENT_COLUMNS].clip(lower=0)))
for column, values in zip(NUTRIENT_COLUMNS, standardized.T):
    recipe_table[f"{column} z"] = values
nutrient_summary = pd.DataFrame({
    "raw median": recipe_table[NUTRIENT_COLUMNS].median(), "raw max": recipe_table[NUTRIENT_COLUMNS].max(),
    "standardized mean": standardized.mean(0), "standardized std": standardized.std(0),
    "standardized min": standardized.min(0), "standardized max": standardized.max(0),
}, index=NUTRIENT_COLUMNS)
display(nutrient_summary.style.format({"raw median": "{:,.0f}", "raw max": "{:,.0f}"} |
                                      {c: "{:.2f}" for c in nutrient_summary.columns if c.startswith("standardized")}))

# Who rated it matters: the same recipe can be positive for one user and negative for another.
rating_rows["label"] = (rating_rows["rating"] >= POSITIVE_MIN).astype(np.int8)
mixed = recipe_groups["rating"].agg(lambda r: (r >= POSITIVE_MIN).any() and (r < POSITIVE_MIN).any())
rated_more = recipe_groups.size() >= 2
print(f"Ratings: {rating_rows['label'].mean():.1%} positive ({POSITIVE_MIN}-5★), {1 - rating_rows['label'].mean():.1%} "
      f"negative (0-{POSITIVE_MIN - 1}★). Of the recipes rated by 2+ users, {mixed[rated_more].mean():.1%} got both a positive "
      f"and a negative rating: the label depends on who rates, not only on the recipe.")


### Positive and negative rating points: negative = 0–3★ vs negative = 0–2★

Every point is **one rating** (a user and a recipe), coloured by its label. A point is placed by its recipe's content, projected to 2-D:
on the left by the **standardized nutrients** (PCA of the seven z-scores), on the right by the **cleaned tokens** (truncated SVD of the
TF-IDF of the recipe's product / adj / verb spans). Both projections are fit on the recipes, each once.

The two rows compare two ways to label the ratings (positive is 4–5★ in both):
- **top: negative = 0–3★**, every rating is used;
- **bottom: negative = 0–2★**, and the 3★ ratings are left out of both classes (neither positive nor negative).

Both rows draw the **same positive points** and the same number of negatives, and share their axes, so the only difference between them
is which ratings count as negative. Negatives are drawn on top. Because the same recipe (the same spot) can be rated positive by one user
and negative by another, the colours overlap: recipe content alone cannot separate the classes. That is why the model below learns a
factor per **user** as well as per recipe.

This is only a view of the data: the models below keep `POSITIVE_MIN` = 4, so they label 0–3★ as negative.


In [ ]:
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
VIZ_STYLE = {
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.titlesize": 12,
    "axes.titleweight": "semibold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
}

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

POINTS_PER_CLASS = 4000

# 2-D projections, fit on the recipes (each once)
nutrient_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit(standardized)
recipe_documents = recipe_table.apply(lambda r: [f"{field}={token}" for field in TOKEN_FIELDS for token in r[f"{field}_tokens"]],
                                      axis=1)
token_matrix = TfidfVectorizer(analyzer=lambda tokens: tokens, min_df=5).fit_transform(recipe_documents)
token_2d = TruncatedSVD(n_components=2, random_state=RANDOM_STATE).fit(token_matrix)
recipe_coords = {
    "nutrients": pd.DataFrame(nutrient_2d.transform(standardized), index=recipe_table.index),
    "tokens": pd.DataFrame(token_2d.transform(token_matrix), index=recipe_table.index),
}

# Two ways to label the ratings (positive = 4-5 stars in both). A rating outside both ranges is left out.
LABEL_SCHEMES = {
    f"negative = 0-{POSITIVE_MIN - 1} stars": range(0, POSITIVE_MIN),       # 0-3: every rating is used
    f"negative = 0-{POSITIVE_MIN - 2} stars, {POSITIVE_MIN - 1} left out": range(0, POSITIVE_MIN - 1),  # 0-2: 3-star ratings dropped
}

# The same positive points in both rows, and the same number of negatives; each point takes its recipe's coordinates.
positive_points = rating_rows[rating_rows["rating"] >= POSITIVE_MIN].sample(n=POINTS_PER_CLASS, random_state=RANDOM_STATE)
with plt.rc_context(VIZ_STYLE):
    fig, axes = plt.subplots(len(LABEL_SCHEMES), 2, figsize=(14, 5.5 * len(LABEL_SCHEMES)), sharex="col", sharey="col")
    for row, (scheme, negative_ratings) in zip(axes, LABEL_SCHEMES.items()):
        is_negative = rating_rows["rating"].isin(negative_ratings)
        used = is_negative | (rating_rows["rating"] >= POSITIVE_MIN)
        negative_share = is_negative.sum() / used.sum()
        negative_points = rating_rows[is_negative].sample(n=POINTS_PER_CLASS, random_state=RANDOM_STATE)
        print(f"{scheme}: {used.sum():,} ratings used ({(~used).sum():,} left out), "
              f"{1 - negative_share:.1%} positive, {negative_share:.1%} negative")
        for ax, (space, coords), axis_names in zip(
                row, recipe_coords.items(),
                [[f"nutrient PC{k} ({ratio:.0%} of variance)" for k, ratio in enumerate(nutrient_2d.explained_variance_ratio_, 1)],
                 [f"token SVD {k}" for k in (1, 2)]]):
            for points, color, name in (
                    (positive_points, BLUE, f"positive ({POSITIVE_MIN}-5 stars), {1 - negative_share:.1%}"),
                    (negative_points, ORANGE, f"negative ({negative_ratings.start}-{negative_ratings.stop - 1} stars), "
                                              f"{negative_share:.1%}")):
                xy = coords.loc[points["recipe_id"]].to_numpy()
                ax.scatter(xy[:, 0], xy[:, 1], s=7, alpha=0.45, color=color, edgecolors="none", label=name)
            ax.set(xlabel=axis_names[0], ylabel=axis_names[1],
                   title=f"{scheme}: ratings by recipe {space} ({POINTS_PER_CLASS:,} points per class)")
            ax.legend(frameon=False, markerscale=3, loc="best", title="share of the ratings used")
    fig.tight_layout()
    plt.show()


## Data: every rating 0–5, positive (4–5★) or negative (0–3★), users with at least 6 ratings, split per user by date


In [ ]:
source = pd.read_csv(DATA_PATH, usecols=["user_id", "recipe_id", "rating", "date"])
rated = source.assign(date=lambda frame: pd.to_datetime(frame["date"], format="mixed"))  # Every rating 0-5 is kept.
# One rating per (user, recipe): the latest.
ratings = rated.sort_values("date", kind="stable").drop_duplicates(["user_id", "recipe_id"], keep="last")
kept = ratings[ratings["user_id"].map(ratings["user_id"].value_counts()) >= MIN_USER_RATINGS]
print(f"{len(source):,} rows (ratings 0-5): {len(rated) - len(ratings):,} repeated (user, recipe) pairs merged -> "
      f"{len(ratings):,} ratings from {ratings['user_id'].nunique():,} users")
print(f"Users with >= {MIN_USER_RATINGS} ratings: {kept['user_id'].nunique():,} kept, with {len(kept):,} ratings on "
      f"{kept['recipe_id'].nunique():,} recipes; {ratings['user_id'].nunique() - kept['user_id'].nunique():,} users "
      f"and their {len(ratings) - len(kept):,} ratings discarded")

# Per user, by date: the latest rating is test, the one before it valid, the rest train. Same-day ties keep file order.
kept = kept.sort_values(["user_id", "date"], kind="stable").reset_index(drop=True)
from_end = kept.groupby("user_id").cumcount(ascending=False).to_numpy()  # 0 = the user's latest rating
positive = kept["rating"].to_numpy() >= POSITIVE_MIN
kept = kept.assign(split=np.select([from_end == 0, from_end == 1], ["test", "valid"], "train"),
                   label=positive.astype(np.int8),
                   source=np.where(positive, f"rated {POSITIVE_MIN}-5", f"rated 0-{POSITIVE_MIN - 1}"))
print(f"Observed ratings: {kept['label'].mean():.1%} positive ({POSITIVE_MIN}-5★), "
      f"{1 - kept['label'].mean():.1%} negative (0-{POSITIVE_MIN - 1}★)")


1,132,367 rows (ratings 0-5): 0 repeated (user, recipe) pairs merged -> 1,132,367 ratings from 226,570 users
Users with >= 6 ratings: 19,665 kept, with 854,916 ratings on 209,476 recipes; 206,905 users and their 277,451 ratings discarded
Observed ratings: 92.2% positive (4-5★), 7.8% negative (0-3★)


## Negative sampling

Every positive rating is paired with 1 recipe (`NEGATIVES_PER_POSITIVE`, so 1 positive : 1 sampled negative) that the user never
rated. They are labelled 0 (negative) and get the positive's split and date. The recipes are drawn uniformly from the recipes of the kept
ratings, without repeats for a user and never one the user rated in any split, so a sampled pair never collides with a real rating and
never appears in two splits.

The table shows the share of positives and negatives in each split, before sampling (observed ratings only) and after it, with the
negatives broken down into real low ratings (0–3★) and sampled unrated recipes.


In [ ]:
def sample_unrated(rng, n, rated_positions, pool_size):
    """n distinct random positions in the recipe pool, none in `rated_positions`: draw, drop rated recipes and repeats, top up."""
    chosen = np.empty(0, dtype=np.int64)
    while len(chosen) < n:
        draw = rng.integers(0, pool_size, size=2 * (n - len(chosen)) + 16)
        chosen = pd.unique(np.concatenate([chosen, draw[~np.isin(draw, rated_positions)]]))  # keeps the draw order
    return chosen[:n]


rng = np.random.default_rng(RANDOM_STATE)
recipe_pool = np.sort(kept["recipe_id"].unique())
rated_positions = np.searchsorted(recipe_pool, kept["recipe_id"].to_numpy())  # every kept recipe is in the pool
rated_rows = kept.groupby("user_id").indices  # user -> rows of all their ratings (every split)
positives = kept[kept["label"] == 1]
negatives_per_positive = np.full(len(positives), NEGATIVES_PER_POSITIVE)

owners, sampled_recipes = [], []  # for each sampled recipe: the row (in `positives`) of the positive it is paired with
for user, rows in positives.groupby("user_id").indices.items():
    counts = negatives_per_positive[rows]
    picked = sample_unrated(rng, counts.sum(), rated_positions[rated_rows[user]], len(recipe_pool))
    owners.append(np.repeat(rows, counts))
    sampled_recipes.append(recipe_pool[picked])
negatives = positives.iloc[np.concatenate(owners)][["user_id", "date", "split"]].assign(
    recipe_id=np.concatenate(sampled_recipes), rating=np.nan, label=np.int8(0), source="sampled (unrated)")
labelled = pd.concat([kept, negatives[kept.columns]], ignore_index=True)
assert not labelled.duplicated(["user_id", "recipe_id"]).any()  # no sampled pair repeats a rating or another sample
print(f"{len(negatives):,} sampled negatives for {len(positives):,} positives "
      f"({len(negatives) / len(positives):.2f} per positive) -> {len(labelled):,} labelled rows")

splits = {name: labelled[labelled["split"] == name].reset_index(drop=True) for name in ("train", "valid", "test")}
train_recipe_ids = set(splits["train"]["recipe_id"])
observed_negative, sampled_negative = f"rated 0-{POSITIVE_MIN - 1}", "sampled (unrated)"
class_balance = pd.DataFrame({
    name: {"rows": len(frame), "users": frame["user_id"].nunique(), "recipes": frame["recipe_id"].nunique(),
           "positive": frame["label"].mean(), "negative": 1 - frame["label"].mean(),
           f"negative: {observed_negative}": frame["source"].eq(observed_negative).mean(),
           f"negative: {sampled_negative}": frame["source"].eq(sampled_negative).mean(),
           "negatives per positive": (frame["label"] == 0).sum() / max(frame["label"].sum(), 1),
           "recipe not in train": (~frame["recipe_id"].isin(train_recipe_ids)).mean()}
    for name, frame in {"observed ratings only": kept, **splits, "all (after sampling)": labelled}.items()}).T
share_columns = ["positive", "negative", f"negative: {observed_negative}", f"negative: {sampled_negative}",
                 "recipe not in train"]
display(class_balance.style.format({"rows": "{:,.0f}", "users": "{:,.0f}", "recipes": "{:,.0f}",
                                    "negatives per positive": "{:.2f}"} | {c: "{:.1%}" for c in share_columns}))
print(f"After sampling: {labelled['label'].mean():.1%} positive, {1 - labelled['label'].mean():.1%} negative "
      f"(before: {kept['label'].mean():.1%} positive, {1 - kept['label'].mean():.1%} negative)")


788,472 sampled negatives for 788,472 positives (1.00 per positive) -> 1,643,388 labelled rows


,rows,users,recipes,positive,negative,negative: rated 0-3,negative: sampled (unrated),negatives per positive,recipe not in train
observed ratings only,"854,916","19,665","209,476",92.2%,7.8%,7.8%,0.0%,0.08,0.0%
train,"1,569,479","19,665","209,364",48.0%,52.0%,3.9%,48.0%,1.08,0.0%
valid,"37,022","19,665","29,247",46.9%,53.1%,6.2%,46.9%,1.13,0.2%
test,"36,887","19,665","29,001",46.7%,53.3%,6.6%,46.7%,1.14,0.2%
all (after sampling),"1,643,388","19,665","209,476",48.0%,52.0%,4.0%,48.0%,1.08,0.0%


After sampling: 48.0% positive, 52.0% negative (before: 92.2% positive, 7.8% negative)


## Label matrix

Users and recipes are indexed from **train** only. `R` is the train label matrix (users × recipes, CSR): 1 for a positive, 0 for a
negative (a 0–3★ rating or a sampled unrated recipe). The zeros are stored explicitly, since they are training rows, and its transpose
(recipes × users) drives the recipe step. A valid or test recipe that has no train row gets index −1.


In [ ]:
user_ids = np.sort(splits["train"]["user_id"].unique())
recipe_ids = np.sort(splits["train"]["recipe_id"].unique())
user_index, recipe_index = pd.Index(user_ids), pd.Index(recipe_ids)


def encode(frame):
    # (user index, recipe index, 0/1 label) arrays; a user or recipe that is not in train gets index -1.
    return (user_index.get_indexer(frame["user_id"]), recipe_index.get_indexer(frame["recipe_id"]),
            frame["label"].to_numpy(np.float64))


encoded = {name: encode(frame) for name, frame in splits.items()}
train_users, train_recipes, train_values = encoded["train"]
R = sp.csr_matrix((train_values, (train_users, train_recipes)), shape=(len(user_ids), len(recipe_ids)))
assert R.nnz == len(splits["train"])  # every train row is stored, the 0 labels included
per_user, per_recipe = np.diff(R.indptr), np.diff(R.tocsc().indptr)
print(f"R: {R.shape[0]:,} users x {R.shape[1]:,} recipes, {R.nnz:,} train rows, {R.sum():,.0f} positive "
      f"(density {R.nnz / np.prod(R.shape):.3%})")
print(f"Train rows per user: min {per_user.min()}, median {np.median(per_user):.0f}, max {per_user.max():,}; "
      f"per recipe: median {np.median(per_recipe):.0f}, max {per_recipe.max():,}, "
      f"{np.mean(per_recipe == 1):.0%} of recipes have one")

# Class weights w_c from the train counts n_c; class_weight[c] is the weight of a row labelled c.
train_counts = np.bincount(train_values.astype(int), minlength=len(LABELS))
class_weight = (len(train_values) / (len(LABELS) * train_counts)) ** WEIGHT_BALANCE
class_weight /= class_weight[train_values.astype(int)].mean()  # The mean train weight is 1.
print(f"Class weights (WEIGHT_BALANCE = {WEIGHT_BALANCE:g}): negative {class_weight[0]:.3f}, "
      f"positive {class_weight[1]:.3f}; weighted train mean {np.average(train_values, weights=class_weight[train_values.astype(int)]):.3f}")


R: 19,665 users x 209,364 recipes, 1,569,479 train rows, 753,893 positive (density 0.038%)
Train rows per user: min 4, median 20, max 15,328; per recipe: median 6, max 954, 1% of recipes have one
Class weights (WEIGHT_BALANCE = 1): negative 0.962, positive 1.041; weighted train mean 0.500


## Focal-loss matrix factorization

- `focal_terms` gives the focal loss of every row and its first and second derivatives with respect to the logit.
- `ridge_rows` solves a weighted ridge regression for every row of a sparse target matrix, meaning every user or every recipe.
  Rows with the same number of entries $c$ are stacked into a `[rows, c, k + 1]` array, so one batched matmul builds their
  $Z^\top \Omega Z$ and one `np.linalg.solve` call solves them.
- `newton_half_step` is one side's update: the Newton step as weighted ridge regressions, then the backtracking line search.
- `fit_mf` alternates the user and recipe steps. After each iteration it records the loss $L$ (and warns if it rises), and the
  log loss and weighted log loss on train and valid. It returns the iteration with the best valid `SELECT_BY`, and it stops after
  `EARLY_STOPPING_PATIENCE` iterations without an improvement.


In [ ]:
def ridge_rows(targets, weights, design, reg):
    """One ALS half-step: a weighted ridge regression for every row of the CSR matrix `targets`.

    With t_r the row's stored values, w_r their weights (`weights` is aligned with targets.data), and
    Z_r = design[the row's columns], row r gets
        x_r = argmin_x sum_j w_rj (t_rj - Z_rj x)^2 + sum_d reg_d * x_d^2 = (Z_r^T W_r Z_r + diag(reg))^-1 Z_r^T W_r t_r.
    """
    d = design.shape[1]
    solution = np.zeros((targets.shape[0], d))  # A row without ratings keeps x = 0.
    counts = np.diff(targets.indptr)
    for c in np.unique(counts[counts > 0]):
        same = np.flatnonzero(counts == c)
        step = max(1, CHUNK_ELEMENTS // ((c + d) * d))  # Rows per batch, to bound memory.
        for start in range(0, len(same), step):
            rows = same[start:start + step]
            at = targets.indptr[rows][:, None] + np.arange(c)  # [rows, c]: where each row's ratings are stored
            Z = design[targets.indices[at]]  # [rows, c, d]
            WZ = Z * weights[at][..., None]  # W_r Z_r
            gram = np.swapaxes(WZ, 1, 2) @ Z + np.diag(reg)  # Z^T W Z + diag(reg)
            rhs = np.einsum("rcd,rc->rd", WZ, targets.data[at])  # Z^T W t
            solution[rows] = np.linalg.solve(gram, rhs[..., None])[..., 0]
    return solution


def logits(model, users, recipes):
    # eta = mu + b_u + c_i + p_u . q_i, in chunks of LOGIT_CHUNK rows. A user or recipe outside train (index -1) adds nothing.
    eta = np.full(len(users), model["mu"])
    known_user, known_recipe = users >= 0, recipes >= 0
    eta[known_user] += model["user_bias"][users[known_user]]
    eta[known_recipe] += model["item_bias"][recipes[known_recipe]]
    both = np.flatnonzero(known_user & known_recipe)
    for start in range(0, len(both), LOGIT_CHUNK):
        rows = both[start:start + LOGIT_CHUNK]
        eta[rows] += np.einsum("nk,nk->n", model["user_factors"][users[rows]], model["item_factors"][recipes[rows]])
    return eta


def predict(model, users, recipes):
    # The probability of a positive: sigmoid of the logit.
    return expit(logits(model, users, recipes))


def focal_terms(eta, y):
    """Focal loss of every row, and its first and second derivatives with respect to the logit eta.

    With s = +1 for a positive and -1 for a negative, u = s * eta and p_t = sigmoid(u) (the probability of the true class):
        FL = -(1 - p_t)^gamma log p_t,     dFL/du = (1 - p_t)^gamma (gamma p_t log p_t - (1 - p_t)),
    and d/deta = s d/du, d2/deta2 = d2/du2. gamma = 0 gives binary cross-entropy: g = p - y, h = p (1 - p).
    """
    s = 2 * y - 1
    p_t, one_minus_p_t, log_p_t = expit(s * eta), expit(-s * eta), log_expit(s * eta)
    modulating = one_minus_p_t ** FOCAL_GAMMA
    inner = FOCAL_GAMMA * p_t * log_p_t - one_minus_p_t
    loss = -modulating * log_p_t
    grad = s * modulating * inner
    hess = modulating * (p_t * one_minus_p_t * (FOCAL_GAMMA * (log_p_t + 1) + 1) - FOCAL_GAMMA * p_t * inner)
    return loss, grad, hess


def probability_metrics(y, p):
    # "log loss": binary cross-entropy of the probability. "weighted log loss": the mean of the per-class log losses, so both
    # classes weigh the same however rare. "focal loss": the unweighted mean focal loss with FOCAL_GAMMA.
    p = np.clip(p, 1e-7, 1 - 1e-7)
    bce = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    label = y.astype(int)
    per_class = [bce[label == c].mean() for c in LABELS if np.any(label == c)]
    return {"log loss": float(bce.mean()), "weighted log loss": float(np.mean(per_class)),
            "focal loss": float(focal_terms(logit(p), y)[0].mean())}


def classification_metrics(y, score):
    # A probability >= DECISION_THRESHOLD is predicted positive: per-class precision / recall / F1, accuracy and
    # macro F1; ROC AUC and average precision (positive class) come from the probability itself.
    label, predicted = y.astype(int), (score >= DECISION_THRESHOLD).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(label, predicted, labels=LABELS, zero_division=0)
    both_classes = len(np.unique(label)) == 2
    return {"accuracy": float(np.mean(predicted == label)), "macro f1": float(f1.mean()),
            **{f"{name} {metric}": float(values[c]) for c, name in enumerate(["negative", "positive"])
               for metric, values in (("precision", precision), ("recall", recall), ("f1", f1))},
            "roc auc": float(roc_auc_score(label, score)) if both_classes else np.nan,
            "average precision": float(average_precision_score(label, score)) if both_classes else np.nan}


SIDES = {"user": ("user_factors", "user_bias", "item_factors", "item_bias"),
         "recipe": ("item_factors", "item_bias", "user_factors", "user_bias")}


def newton_half_step(model, side, M, owners, weight, l2, l2_bias, loss_now, objective):
    """One side's update. `M` is R (side "user") or its transpose (side "recipe"); `owners` is the row of every stored entry.

    Newton step on L for [factors, bias] of every row, as weighted ridge regressions on the working response
    t = eta - mu - (other side's bias) - g / h with weights w h / 2, then a backtracking line search: the first step size
    in LINE_SEARCH_STEPS that does not raise L is kept (none: the side is left as it was).
    """
    own_factors, own_bias, other_factors, other_bias = SIDES[side]
    k = model[own_factors].shape[1]
    users, recipes = (owners, M.indices) if side == "user" else (M.indices, owners)
    eta = logits(model, users, recipes)
    _, grad, hess = focal_terms(eta, M.data)
    hess = np.maximum(hess, MIN_CURVATURE)
    target = eta - model["mu"] - model[other_bias][M.indices] - grad / hess
    design = np.c_[model[other_factors], np.ones(M.shape[1])]
    current = np.c_[model[own_factors], model[own_bias]]
    newton = ridge_rows(sp.csr_matrix((target, M.indices, M.indptr), shape=M.shape), weight * hess / 2, design,
                        np.r_[np.full(k, l2), l2_bias])
    for step in LINE_SEARCH_STEPS:
        candidate = current + step * (newton - current)
        trial = {**model, own_factors: candidate[:, :k], own_bias: candidate[:, k]}
        loss = objective(trial)
        if loss <= loss_now:
            return trial, loss
    return model, loss_now


def fit_mf(R, valid, factors, l2, l2_bias, name, weights=None):
    n_users, n_recipes = R.shape
    Rt = R.T.tocsr()  # Recipes x users, for the recipe step.
    users = np.repeat(np.arange(n_users), np.diff(R.indptr))  # The user of each entry of R.
    recipes_t = np.repeat(np.arange(n_recipes), np.diff(Rt.indptr))  # The recipe of each entry of Rt.
    weights = class_weight if weights is None else weights  # w_c per class; the negative-sampling weights by default
    weight, weight_t = weights[R.data.astype(int)], weights[Rt.data.astype(int)]  # w of each stored label
    model = {"mu": float(logit(np.average(R.data, weights=weight))),
             "user_factors": np.zeros((n_users, factors)), "user_bias": np.zeros(n_users),
             "item_factors": np.random.default_rng(RANDOM_STATE).normal(0, INIT_STD, (n_recipes, factors)),
             "item_bias": np.zeros(n_recipes)}

    def objective(candidate):
        # L: class-weighted focal loss on train plus the L2 penalties.
        focal = focal_terms(logits(candidate, users, R.indices), R.data)[0]
        factor_penalty = (candidate["user_factors"] ** 2).sum() + (candidate["item_factors"] ** 2).sum()
        bias_penalty = (candidate["user_bias"] ** 2).sum() + (candidate["item_bias"] ** 2).sum()
        return float(weight @ focal + l2 * factor_penalty + l2_bias * bias_penalty)

    history, best, best_model, stale, loss = [], np.inf, None, 0, objective(model)
    for iteration in range(1, MAX_ITERATIONS + 1):
        started, previous = time.time(), loss
        model, loss = newton_half_step(model, "user", R, users, weight, l2, l2_bias, loss, objective)
        model, loss = newton_half_step(model, "recipe", Rt, recipes_t, weight_t, l2, l2_bias, loss, objective)
        if loss > previous * (1 + 1e-6):  # The line search only keeps steps that do not raise L.
            warnings.warn(f"{name}: the loss rose at iteration {iteration} ({previous:.6g} -> {loss:.6g}).")
        row = {"iteration": iteration, "loss": loss}
        for split, (split_users, split_recipes, values) in (("train", (users, R.indices, R.data)), ("valid", valid)):
            metrics = probability_metrics(values, predict(model, split_users, split_recipes))
            row.update({f"{split} log loss": metrics["log loss"], f"{split} weighted log loss": metrics["weighted log loss"]})
        history.append({**row, "seconds": time.time() - started})
        if row[f"valid {SELECT_BY}"] < best - MIN_IMPROVEMENT:
            best, best_model, stale = row[f"valid {SELECT_BY}"], {**model, "iterations": iteration}, 0
        else:
            stale += 1
            if stale >= EARLY_STOPPING_PATIENCE:
                break
    history = pd.DataFrame(history)
    best_row = history.iloc[best_model["iterations"] - 1]
    print(f"{name}: valid log loss {best_row['valid log loss']:.4f}, weighted {best_row['valid weighted log loss']:.4f} "
          f"(train {best_row['train log loss']:.4f}, weighted {best_row['train weighted log loss']:.4f}) "
          f"at iteration {best_model['iterations']} of {len(history)}, {history['seconds'].sum():.0f}s")
    return best_model, history


## Choosing $\lambda_b$ and $\lambda$ on valid

1. **Bias baseline** (user + recipe bias, $k = 0$), for every $\lambda_b$ in `L2_BIAS_GRID`. The best one is kept as the baseline and fixes $\lambda_b$.
2. **Focal MF with $k$ = `FACTORS`**, for every $\lambda$ in `L2_GRID`, with that $\lambda_b$.

The best run is the one with the lowest valid `SELECT_BY`. Each table row is the kept iteration of one run.

In [ ]:
METRIC_COLUMNS = [f"{split} {metric}" for split in ("train", "valid") for metric in ("log loss", "weighted log loss")]


def search_table(runs, name):
    # One row per candidate: the kept iteration and its train and valid metrics.
    return pd.DataFrame([{name: value, "kept iteration": model["iterations"], "iterations run": len(history),
                          **history.iloc[model["iterations"] - 1][METRIC_COLUMNS].to_dict()}
                         for value, (model, history) in runs.items()]).set_index(name)


valid_rows = encoded["valid"]
bias_runs = {l2_bias: fit_mf(R, valid_rows, 0, 0.0, l2_bias, f"bias baseline, λ_b={l2_bias:g}")
             for l2_bias in L2_BIAS_GRID}
bias_search = search_table(bias_runs, "λ_b")
best_l2_bias = float(bias_search[f"valid {SELECT_BY}"].idxmin())
als_runs = {l2: fit_mf(R, valid_rows, FACTORS, l2, best_l2_bias, f"focal MF k={FACTORS}, λ={l2:g}") for l2 in L2_GRID}
als_search = search_table(als_runs, "λ")
best_l2 = float(als_search[f"valid {SELECT_BY}"].idxmin())
display(bias_search.round(4), als_search.round(4))
for label, value, grid in (("λ_b", best_l2_bias, L2_BIAS_GRID), ("λ", best_l2, L2_GRID)):
    if value in (min(grid), max(grid)):
        print(f"{label} = {value:g} is at the edge of its grid; extend the grid to check that it is the best.")
print(f"Chosen by valid {SELECT_BY}: λ_b = {best_l2_bias:g}, λ = {best_l2:g}")

bias baseline, λ_b=0.03: valid log loss 0.6245, weighted 0.6248 (train 0.5769, weighted 0.5768) at iteration 2 of 5, 3s
bias baseline, λ_b=0.1: valid log loss 0.6151, weighted 0.6155 (train 0.5755, weighted 0.5752) at iteration 7 of 10, 7s
bias baseline, λ_b=0.3: valid log loss 0.6115, weighted 0.6112 (train 0.5869, weighted 0.5867) at iteration 13 of 16, 11s
bias baseline, λ_b=1: valid log loss 0.6187, weighted 0.6173 (train 0.6088, weighted 0.6084) at iteration 9 of 12, 8s
bias baseline, λ_b=3: valid log loss 0.6352, weighted 0.6331 (train 0.6344, weighted 0.6339) at iteration 5 of 8, 5s
bias baseline, λ_b=10: valid log loss 0.6563, weighted 0.6543 (train 0.6601, weighted 0.6596) at iteration 3 of 6, 4s
bias baseline, λ_b=30: valid log loss 0.6722, weighted 0.6708 (train 0.6762, weighted 0.6758) at iteration 2 of 5, 3s
focal MF k=32, λ=0.1: valid log loss 0.6180, weighted 0.6169 (train 0.1590, weighted 0.1585) at iteration 40 of 40, 170s
focal MF k=32, λ=0.3: valid log loss 0.6012, w

,kept iteration,iterations run,train log loss,train weighted log loss,valid log loss,valid weighted log loss
λ_b,,,,,,
0.03,2,5,0.5769,0.5768,0.6245,0.6248
0.10,7,10,0.5755,0.5752,0.6151,0.6155
0.30,13,16,0.5869,0.5867,0.6115,0.6112
1.00,9,12,0.6088,0.6084,0.6187,0.6173
3.00,5,8,0.6344,0.6339,0.6352,0.6331
10.00,3,6,0.6601,0.6596,0.6563,0.6543
30.00,2,5,0.6762,0.6758,0.6722,0.6708


,kept iteration,iterations run,train log loss,train weighted log loss,valid log loss,valid weighted log loss
λ,,,,,,
0.1,40,40,0.1590,0.1585,0.6180,0.6169
0.3,25,28,0.2422,0.2417,0.6012,0.6006
1.0,13,16,0.3745,0.3741,0.5947,0.5939
3.0,16,19,0.5021,0.5018,0.6029,0.6024
10.0,13,16,0.5821,0.5818,0.6109,0.6106
30.0,13,16,0.5869,0.5867,0.6115,0.6112
100.0,13,16,0.5869,0.5867,0.6115,0.6112


Chosen by valid weighted log loss: λ_b = 0.3, λ = 1


### Train and valid loss by $\lambda$

The loss is `SELECT_BY`, the metric that picks $\lambda$. Where the valid loss rises while the train loss keeps falling
(small $\lambda$), the factors overfit. The dashed line is the bias baseline's valid loss, which the factor model
approaches as $\lambda$ grows.

In [ ]:
metric_label = SELECT_BY

lambdas = als_search.index.to_numpy()
with plt.rc_context(VIZ_STYLE):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.hlines(bias_search.loc[best_l2_bias, f"valid {SELECT_BY}"], lambdas[0], lambdas[-1], colors=MUTED,
              linewidths=1.2, linestyles="--", label=f"bias baseline, valid (λ_b = {best_l2_bias:g})")
    for split, color in (("valid", BLUE), ("train", ORANGE)):
        column = f"{split} {SELECT_BY}"
        ax.plot(lambdas, als_search[column], color=color, lw=2, marker="o", ms=8, mec=SURFACE, mew=1.5,
                solid_capstyle="round", solid_joinstyle="round", label=f"focal MF, {split}")
        ax.annotate(split, (lambdas[-1], als_search[column].iat[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color=INK_2)
    # The chosen lambda: a ring around its valid point.
    ax.plot(best_l2, als_search.loc[best_l2, f"valid {SELECT_BY}"], marker="o", ms=15, mfc="none", mec=INK, mew=1.2)
    ax.set_xscale("log")
    ax.set_xticks(lambdas, [f"{value:g}" for value in lambdas])
    ax.minorticks_off()
    ax.set(xlabel="λ (L2 penalty on the factors)", ylabel=metric_label,
           title=f"Focal MF, k = {FACTORS}: {metric_label} at the kept iteration (ring: chosen λ = {best_l2:g})")
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

## Test

The chosen model next to the two baselines: the **base rate** $\sigma(\mu)$, a constant probability (so it predicts every row as the same
class and has ROC AUC 0.5), and the **user + recipe bias** model (no factors).
Every model is scored per class (negative and positive) at `DECISION_THRESHOLD`, plus accuracy, macro F1, ROC AUC and average precision,
and the log losses of the probability. `test, recipe in train` keeps only the test rows whose recipe has a train row. On the other rows every model
gets its logit from $\mu$ and $b_u$ alone, so this subset shows what the factors add.


In [ ]:
bias_model, als_model = bias_runs[best_l2_bias][0], als_runs[best_l2][0]
test_users, test_recipes, test_values = encoded["test"]
in_train = test_recipes >= 0
eval_rows = {"valid": encoded["valid"], "test": encoded["test"],
             "test, recipe in train": (test_users[in_train], test_recipes[in_train], test_values[in_train])}
models = {
    f"baseline: base rate (constant {expit(als_model['mu']):.2f})": lambda users, recipes: np.full(len(users), expit(als_model["mu"])),
    f"baseline: user + recipe bias (λ_b={best_l2_bias:g})": lambda users, recipes: predict(bias_model, users, recipes),
    f"focal MF k={FACTORS} (γ={FOCAL_GAMMA:g}, λ={best_l2:g}, λ_b={best_l2_bias:g})":
        lambda users, recipes: predict(als_model, users, recipes),
}
result_rows = {}
for model, score_rows in models.items():
    for split, (users, recipes, values) in eval_rows.items():
        score = score_rows(users, recipes)
        result_rows[(model, split)] = {**classification_metrics(values, score), **probability_metrics(values, score)}
results = pd.DataFrame(result_rows).T.rename_axis(["model", "split"])
display(results.round(4))
print(f"{(~in_train).mean():.1%} of test rows have a recipe with no train row.")


accuracy  \
model                                  split                             
baseline: base rate (constant 0.50)    valid                    0.4688   
                                       test                     0.4669   
                                       test, recipe in train    0.4664   
baseline: user + recipe bias (λ_b=0.3) valid                    0.7002   
                                       test                     0.6946   
                                       test, recipe in train    0.6952   
focal MF k=32 (γ=2, λ=1, λ_b=0.3)      valid                    0.7163   
                                       test                     0.7056   
                                       test, recipe in train    0.7063   

                                                              macro f1  \
model                                  split                             
baseline: base rate (constant 0.50)    valid                    0.3192   
                                       test                     0.3183   
                                       test, recipe in train    0.3180   
baseline: user + recipe bias (λ_b=0.3) valid                    0.6990   
                                       test                     0.6931   
                                       test, recipe in train    0.6937   
focal MF k=32 (γ=2, λ=1, λ_b=0.3)      valid                    0.7143   
                                       test                     0.7033   
                                       test, recipe in train    0.7040   

                                                              negative precision  \
model                                  split                                       
baseline: base rate (constant 0.50)    valid                              0.0000   
                                       test                               0.0000   
                                       test, recipe in train              0.0000   
baseline: user + recipe bias (λ_b=0.3) valid                              0.7167   
                                       test                               0.7122   
                                       test, recipe in train              0.7135   
focal MF k=32 (γ=2, λ=1, λ_b=0.3)      valid                              0.7234   
                                       test                               0.7152   
                                       test, recipe in train              0.7166   

                                                              negative recall  \
model                                  split                                    
baseline: base rate (constant 0.50)    valid                           0.0000   
                                       test                            0.0000   
                                       test, recipe in train           0.0000   
baseline: user + recipe bias (λ_b=0.3) valid                           0.7205   
                                       test                            0.7166   
                                       test, recipe in train           0.7164   
focal MF k=32 (γ=2, λ=1, λ_b=0.3)      valid                           0.7544   
                                       test                            0.7440   
                                       test, recipe in train           0.7439   

                                                              negative f1  \
model                                  split                                
baseline: base rate (constant 0.50)    valid                       0.0000   
                                       test                        0.0000   
                                       test, recipe in train       0.0000   
baseline: user + recipe bias (λ_b=0.3) valid                       0.7186   
                                       test                        0.7144   
                                       test, recipe in train       0.7150   

0.2% of test rows have a recipe with no train row.


### How well each class is predicted

The chosen focal MF model on test, at `DECISION_THRESHOLD`: for each class, **precision** is the share of the rows predicted as that class
that belong to it, **recall** is the share of the class's rows predicted as it, and **F1** is their harmonic mean. The fourth bar, also
printed under each group, is the class's share of all labelled rows. The confusion matrix gives the counts, and the last table splits the
negatives into real low ratings (0–3★) and sampled unrated recipes, to show which kind the model catches.


In [ ]:
class_names = ["negative", "positive"]
true_label = test_values.astype(int)
test_score = predict(als_model, test_users, test_recipes)
predicted_label = (test_score >= DECISION_THRESHOLD).astype(int)
precision, recall, f1, support = precision_recall_fscore_support(true_label, predicted_label, labels=LABELS,
                                                                 zero_division=0)
per_class = pd.DataFrame({
    "precision": precision, "recall": recall, "f1": f1,
    "share of dataset": labelled["label"].value_counts(normalize=True).reindex(LABELS, fill_value=0).to_numpy(),
    "test rows": support, "predicted": np.bincount(predicted_label, minlength=len(LABELS)),
}, index=pd.Index(class_names, name="class"))
display(per_class.round(3))
display(pd.DataFrame(confusion_matrix(true_label, predicted_label, labels=LABELS),
                     index=[f"true {name}" for name in class_names],
                     columns=[f"predicted {name}" for name in class_names]))
test_source = splits["test"]["source"].to_numpy()
by_source = pd.DataFrame({
    kind: {"test rows": int(rows.sum()), "predicted positive": predicted_label[rows].mean(),
           "mean score": test_score[rows].mean()}
    for kind in pd.unique(test_source) for rows in [test_source == kind]}).T.rename_axis("test rows by source")
display(by_source.round(3))
test_metrics = classification_metrics(test_values, test_score)
print(f"Test accuracy {test_metrics['accuracy']:.3f}, macro F1 {test_metrics['macro f1']:.3f}, "
      f"ROC AUC {test_metrics['roc auc']:.3f}, average precision {test_metrics['average precision']:.3f} "
      f"(threshold {DECISION_THRESHOLD:g})")

positions = np.arange(len(LABELS))
bars = (("precision", BLUE, "precision"), ("recall", ORANGE, "recall"), ("f1", AQUA, "F1"),
        ("share of dataset", YELLOW, "share of the dataset"))
width, spacing = 0.15, 0.165  # Bar width, and the distance between bar centers (a thin gap between neighbours).
class_labels = [f"negative (0-{POSITIVE_MIN - 1} stars or unrated)", f"positive ({POSITIVE_MIN}-5 stars)"]
with plt.rc_context(VIZ_STYLE):
    fig, ax = plt.subplots(figsize=(8, 4.4))
    for k, (column, color, label) in enumerate(bars):
        ax.bar(positions + (k - (len(bars) - 1) / 2) * spacing, per_class[column], width=width, color=color, label=label)
    ax.set_xticks(positions, [f"{name}\n{share:.1%} of rows" for name, share in zip(class_labels, per_class["share of dataset"])])
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set(ylim=(0, 1.2), xlabel="class", ylabel="score or share",
           title=f"Focal MF on test: precision, recall and F1 by class (γ = {FOCAL_GAMMA:g}, WEIGHT_BALANCE = {WEIGHT_BALANCE:g})")
    ax.tick_params(axis="x", length=0)
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False, ncols=4, loc="upper left")
    fig.tight_layout()
    plt.show()


## Save the model

`model_artifacts/mf_als/` holds:

- `als_model.npz`: `mu`; `user_bias` and `user_factors`, one row per entry of `user_ids`; `item_bias` and `item_factors`, one row per entry of `recipe_ids`.
- `l2_search.csv` (both searches), `metrics.csv` (the test table), `per_class.csv` (precision, recall, and F1 per class), and
  `config.json` (data filters, labels, negative sampling, split, class weights, focal loss, threshold, and hyperparameters).

To score user $u$ on recipe $i$, add $\mu$, $b_u$, $c_i$, and $p_u^\top q_i$ (the logit), leaving out the terms of a user or recipe that
is not in the vocabulary, then apply the sigmoid. A probability of at least `DECISION_THRESHOLD` means positive (the user likes the recipe).


In [ ]:
ALS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(ALS_DIR / "als_model.npz", user_ids=user_ids, recipe_ids=recipe_ids, mu=als_model["mu"],
         user_bias=als_model["user_bias"], user_factors=als_model["user_factors"],
         item_bias=als_model["item_bias"], item_factors=als_model["item_factors"])
pd.concat({"bias baseline": bias_search.rename_axis("l2"), f"focal MF k={FACTORS}": als_search.rename_axis("l2")},
          names=["model"]).to_csv(ALS_DIR / "l2_search.csv")
results.to_csv(ALS_DIR / "metrics.csv")
per_class.to_csv(ALS_DIR / "per_class.csv")
(ALS_DIR / "config.json").write_text(json.dumps({
    "task": "binary: positive (rating >= positive_min) vs negative; logistic biased matrix factorization",
    "loss": "class-weighted focal loss (1 - p_t)^gamma * BCE + L2, fitted by alternating Newton steps with a line search",
    "score": "sigmoid(mu + b_u + c_i + p_u . q_i)", "focal_gamma": FOCAL_GAMMA,
    "data": str(DATA_PATH.relative_to(NOTEBOOK_DIR)), "ratings_kept": "0-5",
    "positive_min": POSITIVE_MIN, "labels": {"0": f"negative: rating 0-{POSITIVE_MIN - 1} or a sampled unrated recipe",
                                             "1": f"positive: rating {POSITIVE_MIN}-5"},
    "negative_sampling": {"per_positive": NEGATIVES_PER_POSITIVE,
                          "pool": "recipes of the kept ratings, uniform, never rated by the user, no repeats per user",
                          "split": "the positive's split"},
    "min_user_ratings": MIN_USER_RATINGS,
    "split": "per user by date: latest rating -> test, the one before -> valid, the rest -> train",
    "users": len(user_ids), "recipes": len(recipe_ids), "train_rows": int(R.nnz), "train_positive": int(R.sum()),
    "weight_balance": WEIGHT_BALANCE, "class_weights": {str(c): float(class_weight[c]) for c in LABELS},
    "decision_threshold": DECISION_THRESHOLD, "min_curvature": MIN_CURVATURE, "line_search_steps": list(LINE_SEARCH_STEPS),
    "select_by": SELECT_BY, "factors": FACTORS, "l2": best_l2, "l2_bias": best_l2_bias,
    "l2_grid": L2_GRID, "l2_bias_grid": L2_BIAS_GRID, "iterations": int(als_model["iterations"]),
    "init_std": INIT_STD, "random_state": RANDOM_STATE,
}, indent=2))
print(f"Saved to {ALS_DIR}: {sorted(p.name for p in ALS_DIR.iterdir())}")


Saved to /Users/tung/Desktop/Học/Projects/ProjectSem1_2/DataMappingRecommendation/model_artifacts/mf_als: ['als_model.npz', 'config.json', 'l2_search.csv', 'metrics.csv', 'per_class.csv', 'per_rating.csv']


## Undersampling: One-Sided Selection

The second approach adds **no sampled negatives**. It trains on the observed ratings only, where the classes are imbalanced the other way:
about 92% of the ratings are positive (4–5★). **One-Sided Selection** (OSS; Kubat & Matwin, 1997) undersamples the majority class,
here the **positive** train rows, in two steps:

1. **Condensed nearest neighbour** (one pass): $C$ = every minority (negative) row plus `OSS_SEEDS` random majority rows. A 1-nearest-neighbour
   classifier fit on $C$ labels every other majority row. The ones it gets right are redundant (far from the class boundary) and are
   removed; the misclassified ones are kept.
2. **Tomek links**: two rows of different classes that are each other's nearest neighbour. The majority row of every such pair
   (borderline or noisy) is removed.

This is imbalanced-learn's `OneSidedSelection` algorithm, written here with scikit-learn's nearest-neighbour search. OSS needs a feature
vector per row, and the factorization has none, so each train (user, recipe) row is described by **leave-one-out statistics** of its user
and of its recipe: mean rating, positive rate and log count, computed without the row itself (so a row's features never contain its own
label), then standardized. Only **train** is undersampled; valid and test keep every observed rating.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler

OSS_DIR = NOTEBOOK_DIR / "model_artifacts/mf_als_one_sided_selection"
OSS_SEEDS = 1  # Majority rows put in C with every minority row (imbalanced-learn's n_seeds_S). More seeds remove more rows.

observed = {name: kept[kept["split"] == name].reset_index(drop=True) for name in ("train", "valid", "test")}
train_observed = observed["train"]


def leave_one_out_stats(frame, key):
    """Mean rating, positive rate and log count of each row's user or recipe (`key`), leaving the row itself out.
    A user or recipe with no other row gets the train mean rating and positive rate."""
    group = frame.groupby(key)
    others = group["rating"].transform("size").to_numpy() - 1
    rating_sum = group["rating"].transform("sum").to_numpy() - frame["rating"].to_numpy()
    positive_sum = group["label"].transform("sum").to_numpy() - frame["label"].to_numpy()
    has_others, denominator = others > 0, np.maximum(others, 1)
    return np.c_[np.where(has_others, rating_sum / denominator, frame["rating"].mean()),
                 np.where(has_others, positive_sum / denominator, frame["label"].mean()),
                 np.log1p(others)]


def one_sided_selection(features, y, n_seeds, seed):
    """One-Sided Selection: returns the kept row positions and how many majority rows each step removed."""
    majority = int(np.bincount(y).argmax())
    majority_rows, minority_rows = np.flatnonzero(y == majority), np.flatnonzero(y != majority)
    seeds = np.random.default_rng(seed).choice(majority_rows, size=n_seeds, replace=False)
    # 1. condensed nearest neighbour: keep only the majority rows a 1-NN fit on C misclassifies
    C, others = np.r_[minority_rows, seeds], np.setdiff1d(majority_rows, seeds)
    one_nn = KNeighborsClassifier(n_neighbors=1, n_jobs=-1).fit(features[C], y[C])
    misclassified = others[one_nn.predict(features[others]) != majority]
    after_cnn = np.sort(np.r_[C, misclassified])
    # 2. Tomek links: mutual nearest neighbours of different classes; drop their majority rows
    nearest = NearestNeighbors(n_neighbors=2, n_jobs=-1).fit(features[after_cnn]).kneighbors(
        features[after_cnn], return_distance=False)[:, 1]
    labels = y[after_cnn]
    tomek = (labels != labels[nearest]) & (nearest[nearest] == np.arange(len(after_cnn)))
    drop = tomek & (labels == majority)
    return after_cnn[~drop], {"majority class": ["negative", "positive"][majority],
                              "removed by condensed nearest neighbour": len(others) - len(misclassified),
                              "removed by Tomek links": int(drop.sum())}


started = time.time()
oss_features = StandardScaler().fit_transform(np.hstack([leave_one_out_stats(train_observed, "user_id"),
                                                         leave_one_out_stats(train_observed, "recipe_id")]))
oss_rows, oss_steps = one_sided_selection(oss_features, train_observed["label"].to_numpy(), OSS_SEEDS, RANDOM_STATE)
oss_splits = {"train": train_observed.iloc[oss_rows].reset_index(drop=True),
              "valid": observed["valid"], "test": observed["test"]}
print(f"One-Sided Selection in {time.time() - started:.0f}s: removed {len(train_observed) - len(oss_rows):,} of "
      f"{len(train_observed):,} train rows ({oss_steps})")

oss_balance = pd.DataFrame({
    name: {"rows": len(frame), "users": frame["user_id"].nunique(), "recipes": frame["recipe_id"].nunique(),
           "positive": frame["label"].mean(), "negative": 1 - frame["label"].mean(),
           "negatives per positive": (frame["label"] == 0).sum() / max(frame["label"].sum(), 1)}
    for name, frame in {"train before OSS": train_observed, "train after OSS": oss_splits["train"],
                        "valid": oss_splits["valid"], "test": oss_splits["test"]}.items()}).T
display(oss_balance.style.format({"rows": "{:,.0f}", "users": "{:,.0f}", "recipes": "{:,.0f}", "positive": "{:.1%}",
                                  "negative": "{:.1%}", "negatives per positive": "{:.3f}"}))
print(f"Train after OSS: {oss_splits['train']['label'].mean():.1%} positive, {1 - oss_splits['train']['label'].mean():.1%} negative "
      f"(before: {train_observed['label'].mean():.1%} positive, {1 - train_observed['label'].mean():.1%} negative)")


One-Sided Selection in 4s: removed 14,547 of 815,586 train rows ({'majority class': 'positive', 'removed by condensed nearest neighbour': 15, 'removed by Tomek links': 14532})


,rows,users,recipes,positive,negative,negatives per positive
train before OSS,"815,586","19,665","205,794",92.4%,7.6%,0.082
train after OSS,"801,039","19,665","205,694",92.3%,7.7%,0.083
valid,"19,665","19,665","13,671",88.3%,11.7%,0.133
test,"19,665","19,665","13,544",87.6%,12.4%,0.142


Train after OSS: 92.3% positive, 7.7% negative (before: 92.4% positive, 7.6% negative)


### Label matrix, class weights and $\lambda$ search (One-Sided Selection)

The same focal-loss matrix factorization and the same two-stage search as for negative sampling, on the undersampled train rows.
Users and recipes are indexed from this train set, and the class weights come from its own class counts (`WEIGHT_BALANCE`).


In [ ]:
oss_user_ids = np.sort(oss_splits["train"]["user_id"].unique())
oss_recipe_ids = np.sort(oss_splits["train"]["recipe_id"].unique())
oss_user_index, oss_recipe_index = pd.Index(oss_user_ids), pd.Index(oss_recipe_ids)


def encode_oss(frame):
    # (user index, recipe index, 0/1 label) in the One-Sided Selection vocabulary; unknown users or recipes get -1.
    return (oss_user_index.get_indexer(frame["user_id"]), oss_recipe_index.get_indexer(frame["recipe_id"]),
            frame["label"].to_numpy(np.float64))


oss_encoded = {name: encode_oss(frame) for name, frame in oss_splits.items()}
oss_train_users, oss_train_recipes, oss_train_values = oss_encoded["train"]
R_oss = sp.csr_matrix((oss_train_values, (oss_train_users, oss_train_recipes)), shape=(len(oss_user_ids), len(oss_recipe_ids)))
assert R_oss.nnz == len(oss_splits["train"])  # every train row is stored, the 0 labels included
oss_counts = np.bincount(oss_train_values.astype(int), minlength=len(LABELS))
oss_class_weight = (len(oss_train_values) / (len(LABELS) * oss_counts)) ** WEIGHT_BALANCE
oss_class_weight /= oss_class_weight[oss_train_values.astype(int)].mean()  # The mean train weight is 1.
print(f"R (OSS): {R_oss.shape[0]:,} users x {R_oss.shape[1]:,} recipes, {R_oss.nnz:,} train rows, {R_oss.sum():,.0f} positive; "
      f"class weights negative {oss_class_weight[0]:.3f}, positive {oss_class_weight[1]:.3f}")

oss_valid = oss_encoded["valid"]
oss_bias_runs = {l2_bias: fit_mf(R_oss, oss_valid, 0, 0.0, l2_bias, f"OSS bias baseline, λ_b={l2_bias:g}",
                                 weights=oss_class_weight) for l2_bias in L2_BIAS_GRID}
oss_bias_search = search_table(oss_bias_runs, "λ_b")
oss_best_l2_bias = float(oss_bias_search[f"valid {SELECT_BY}"].idxmin())
oss_als_runs = {l2: fit_mf(R_oss, oss_valid, FACTORS, l2, oss_best_l2_bias, f"OSS focal MF k={FACTORS}, λ={l2:g}",
                           weights=oss_class_weight) for l2 in L2_GRID}
oss_als_search = search_table(oss_als_runs, "λ")
oss_best_l2 = float(oss_als_search[f"valid {SELECT_BY}"].idxmin())
display(oss_bias_search.round(4), oss_als_search.round(4))
for label, value, grid in (("λ_b", oss_best_l2_bias, L2_BIAS_GRID), ("λ", oss_best_l2, L2_GRID)):
    if value in (min(grid), max(grid)):
        print(f"{label} = {value:g} is at the edge of its grid; extend the grid to check that it is the best.")
print(f"Chosen by valid {SELECT_BY}: λ_b = {oss_best_l2_bias:g}, λ = {oss_best_l2:g}")


R (OSS): 19,665 users x 205,694 recipes, 801,039 train rows, 739,346 positive; class weights negative 6.492, positive 0.542
OSS bias baseline, λ_b=0.03: valid log loss 0.5757, weighted 0.6710 (train 0.4720, weighted 0.4669) at iteration 1 of 4, 1s
OSS bias baseline, λ_b=0.1: valid log loss 0.5880, weighted 0.6645 (train 0.4915, weighted 0.4801) at iteration 1 of 4, 1s
OSS bias baseline, λ_b=0.3: valid log loss 0.5967, weighted 0.6569 (train 0.4783, weighted 0.4653) at iteration 2 of 5, 2s
OSS bias baseline, λ_b=1: valid log loss 0.6338, weighted 0.6527 (train 0.5144, weighted 0.5091) at iteration 3 of 6, 2s
OSS bias baseline, λ_b=3: valid log loss 0.6612, weighted 0.6601 (train 0.5611, weighted 0.5594) at iteration 3 of 6, 2s
OSS bias baseline, λ_b=10: valid log loss 0.6795, weighted 0.6724 (train 0.6032, weighted 0.6046) at iteration 2 of 5, 2s
OSS bias baseline, λ_b=30: valid log loss 0.6876, weighted 0.6817 (train 0.6282, weighted 0.6337) at iteration 2 of 5, 2s
OSS focal MF k=32, λ

,kept iteration,iterations run,train log loss,train weighted log loss,valid log loss,valid weighted log loss
λ_b,,,,,,
0.03,1,4,0.4720,0.4669,0.5757,0.6710
0.10,1,4,0.4915,0.4801,0.5880,0.6645
0.30,2,5,0.4783,0.4653,0.5967,0.6569
1.00,3,6,0.5144,0.5091,0.6338,0.6527
3.00,3,6,0.5611,0.5594,0.6612,0.6601
10.00,2,5,0.6032,0.6046,0.6795,0.6724
30.00,2,5,0.6282,0.6337,0.6876,0.6817


,kept iteration,iterations run,train log loss,train weighted log loss,valid log loss,valid weighted log loss
λ,,,,,,
0.1,1,4,0.3666,0.3472,0.5778,0.6826
0.3,1,4,0.4197,0.3939,0.5800,0.6694
1.0,1,4,0.5019,0.4780,0.6071,0.6586
3.0,3,6,0.4733,0.4681,0.6228,0.6532
10.0,3,6,0.5139,0.5089,0.6336,0.6528
30.0,3,6,0.5142,0.5090,0.6337,0.6527
100.0,3,6,0.5144,0.5090,0.6337,0.6527


λ = 100 is at the edge of its grid; extend the grid to check that it is the best.
Chosen by valid weighted log loss: λ_b = 1, λ = 100


### Test (One-Sided Selection), and both approaches on the same rows

The chosen models on the observed valid and test ratings, per class at `DECISION_THRESHOLD`, with the confusion matrix.
The last table puts **both approaches on the same rows**: the observed test ratings, which are the negative-sampling test set
without its sampled negatives (each model scores them with its own vocabulary).


In [ ]:
oss_bias_model, oss_model = oss_bias_runs[oss_best_l2_bias][0], oss_als_runs[oss_best_l2][0]
oss_models = {
    f"baseline: base rate (constant {expit(oss_model['mu']):.2f})": lambda users, recipes: np.full(len(users), expit(oss_model["mu"])),
    f"baseline: user + recipe bias (λ_b={oss_best_l2_bias:g})": lambda users, recipes: predict(oss_bias_model, users, recipes),
    f"focal MF k={FACTORS} (γ={FOCAL_GAMMA:g}, λ={oss_best_l2:g}, λ_b={oss_best_l2_bias:g})":
        lambda users, recipes: predict(oss_model, users, recipes),
}
oss_result_rows = {}
for model, score_rows in oss_models.items():
    for split in ("valid", "test"):
        users, recipes, values = oss_encoded[split]
        score = score_rows(users, recipes)
        oss_result_rows[(model, split)] = {**classification_metrics(values, score), **probability_metrics(values, score)}
oss_results = pd.DataFrame(oss_result_rows).T.rename_axis(["model", "split"])
display(oss_results.round(4))

oss_test_users, oss_test_recipes, oss_test_values = oss_encoded["test"]
oss_test_score = predict(oss_model, oss_test_users, oss_test_recipes)
oss_true, oss_predicted = oss_test_values.astype(int), (oss_test_score >= DECISION_THRESHOLD).astype(int)
precision, recall, f1, support = precision_recall_fscore_support(oss_true, oss_predicted, labels=LABELS, zero_division=0)
oss_per_class = pd.DataFrame({
    "precision": precision, "recall": recall, "f1": f1,
    "share of train after OSS": oss_splits["train"]["label"].value_counts(normalize=True).reindex(LABELS, fill_value=0).to_numpy(),
    "test rows": support, "predicted": np.bincount(oss_predicted, minlength=len(LABELS)),
}, index=pd.Index(class_names, name="class"))
display(oss_per_class.round(3))
display(pd.DataFrame(confusion_matrix(oss_true, oss_predicted, labels=LABELS),
                     index=[f"true {name}" for name in class_names], columns=[f"predicted {name}" for name in class_names]))

# both approaches on the observed test ratings
ns_users, ns_recipes, ns_values = encode(observed["test"])  # the negative-sampling vocabulary
ns_score = predict(als_model, ns_users, ns_recipes)
approach_comparison = pd.DataFrame({
    f"negative sampling (1:{NEGATIVES_PER_POSITIVE})": {**classification_metrics(ns_values, ns_score),
                                                        **probability_metrics(ns_values, ns_score)},
    f"undersampling (One-Sided Selection)": {**classification_metrics(oss_test_values, oss_test_score),
                                             **probability_metrics(oss_test_values, oss_test_score)},
}).T.rename_axis(f"observed test ratings ({len(oss_test_values):,} rows)")
display(approach_comparison.round(4))


accuracy  macro f1  \
model                                split                       
baseline: base rate (constant 0.50)  valid    0.1174    0.1050   
                                     test     0.1242    0.1105   
baseline: user + recipe bias (λ_b=1) valid    0.6578    0.5274   
                                     test     0.6511    0.5247   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid    0.6578    0.5274   
                                     test     0.6512    0.5246   

                                            negative precision  \
model                                split                       
baseline: base rate (constant 0.50)  valid              0.1174   
                                     test               0.1242   
baseline: user + recipe bias (λ_b=1) valid              0.1854   
                                     test               0.1880   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid              0.1854   
                                     test               0.1879   

                                            negative recall  negative f1  \
model                                split                                 
baseline: base rate (constant 0.50)  valid           1.0000       0.2101   
                                     test            1.0000       0.2210   
baseline: user + recipe bias (λ_b=1) valid           0.5646       0.2791   
                                     test            0.5448       0.2795   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid           0.5646       0.2792   
                                     test            0.5444       0.2794   

                                            positive precision  \
model                                split                       
baseline: base rate (constant 0.50)  valid              0.0000   
                                     test               0.0000   
baseline: user + recipe bias (λ_b=1) valid              0.9205   
                                     test               0.9116   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid              0.9205   
                                     test               0.9116   

                                            positive recall  positive f1  \
model                                split                                 
baseline: base rate (constant 0.50)  valid           0.0000       0.0000   
                                     test            0.0000       0.0000   
baseline: user + recipe bias (λ_b=1) valid           0.6702       0.7756   
                                     test            0.6662       0.7698   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid           0.6702       0.7757   
                                     test            0.6663       0.7699   

                                            roc auc  average precision  \
model                                split                               
baseline: base rate (constant 0.50)  valid   0.5000             0.8826   
                                     test    0.5000             0.8758   
baseline: user + recipe bias (λ_b=1) valid   0.6644             0.9296   
                                     test    0.6467             0.9190   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid   0.6643             0.9296   
                                     test    0.6467             0.9190   

                                            log loss  weighted log loss  \
model                                split                                
baseline: base rate (constant 0.50)  valid    0.6931             0.6931   
                                     test     0.6931             0.6931   
baseline: user + recipe bias (λ_b=1) valid    0.6338             0.6527   
                                     test     0.6363             0.6593   
focal MF k=32 (γ=2, λ=100, λ_b=1)    valid    0.6337             0.6527   
                                     test     0.6363             0.6593   

                                            f

,precision,recall,f1,share of train after OSS,test rows,predicted
class,,,,,,
negative,0.188,0.544,0.279,0.077,2443,7077
positive,0.912,0.666,0.770,0.923,17222,12588


,predicted negative,predicted positive
true negative,1330,1113
true positive,5747,11475


,accuracy,macro f1,negative precision,negative recall,negative f1,positive precision,positive recall,positive f1,roc auc,average precision,log loss,weighted log loss,focal loss
"observed test ratings (19,665 rows)",,,,,,,,,,,,,
negative sampling (1:1),0.6311,0.4885,0.1483,0.4151,0.2185,0.8886,0.6618,0.7586,0.5578,0.8949,0.6207,0.7186,0.1764
undersampling (One-Sided Selection),0.6512,0.5246,0.1879,0.5444,0.2794,0.9116,0.6663,0.7699,0.6467,0.9190,0.6363,0.6593,0.1624


### Save the model (One-Sided Selection)

`model_artifacts/mf_als_one_sided_selection/` holds the same files as the negative-sampling model (`als_model.npz`, `l2_search.csv`,
`metrics.csv`, `per_class.csv`, `config.json`), plus `class_balance.csv` (the train balance before and after OSS) and
`approach_comparison.csv` (both approaches on the observed test ratings). Scoring works the same way: the sigmoid of
$\mu + b_u + c_i + p_u^\top q_i$ with this model's `user_ids` and `recipe_ids`, positive at `DECISION_THRESHOLD`.


In [ ]:
OSS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(OSS_DIR / "als_model.npz", user_ids=oss_user_ids, recipe_ids=oss_recipe_ids, mu=oss_model["mu"],
         user_bias=oss_model["user_bias"], user_factors=oss_model["user_factors"],
         item_bias=oss_model["item_bias"], item_factors=oss_model["item_factors"])
pd.concat({"bias baseline": oss_bias_search.rename_axis("l2"), f"focal MF k={FACTORS}": oss_als_search.rename_axis("l2")},
          names=["model"]).to_csv(OSS_DIR / "l2_search.csv")
oss_results.to_csv(OSS_DIR / "metrics.csv")
oss_per_class.to_csv(OSS_DIR / "per_class.csv")
oss_balance.to_csv(OSS_DIR / "class_balance.csv")
approach_comparison.to_csv(OSS_DIR / "approach_comparison.csv")
(OSS_DIR / "config.json").write_text(json.dumps({
    "task": "binary: positive (rating >= positive_min) vs negative; logistic biased matrix factorization",
    "imbalance": "undersampling with One-Sided Selection (condensed nearest neighbour + Tomek links) on the observed train ratings",
    "one_sided_selection": {"seeds": OSS_SEEDS, "steps": oss_steps,
                            "features": "leave-one-out user and recipe mean rating, positive rate and log count, standardized",
                            "train_rows_before": len(train_observed), "train_rows_after": len(oss_splits["train"])},
    "loss": "class-weighted focal loss (1 - p_t)^gamma * BCE + L2, fitted by alternating Newton steps with a line search",
    "score": "sigmoid(mu + b_u + c_i + p_u . q_i)", "focal_gamma": FOCAL_GAMMA,
    "data": str(DATA_PATH.relative_to(NOTEBOOK_DIR)), "ratings_kept": "0-5",
    "positive_min": POSITIVE_MIN, "labels": {"0": f"negative: rating 0-{POSITIVE_MIN - 1}", "1": f"positive: rating {POSITIVE_MIN}-5"},
    "min_user_ratings": MIN_USER_RATINGS,
    "split": "per user by date: latest rating -> test, the one before -> valid, the rest -> train (only train is undersampled)",
    "users": len(oss_user_ids), "recipes": len(oss_recipe_ids), "train_rows": int(R_oss.nnz), "train_positive": int(R_oss.sum()),
    "weight_balance": WEIGHT_BALANCE, "class_weights": {str(c): float(oss_class_weight[c]) for c in LABELS},
    "decision_threshold": DECISION_THRESHOLD, "min_curvature": MIN_CURVATURE, "line_search_steps": list(LINE_SEARCH_STEPS),
    "select_by": SELECT_BY, "factors": FACTORS, "l2": oss_best_l2, "l2_bias": oss_best_l2_bias,
    "l2_grid": L2_GRID, "l2_bias_grid": L2_BIAS_GRID, "iterations": int(oss_model["iterations"]),
    "init_std": INIT_STD, "random_state": RANDOM_STATE,
}, indent=2))
print(f"Saved to {OSS_DIR}: {sorted(p.name for p in OSS_DIR.iterdir())}")


Saved to /Users/tung/Desktop/Học/Projects/ProjectSem1_2/DataMappingRecommendation/model_artifacts/mf_als_one_sided_selection: ['als_model.npz', 'approach_comparison.csv', 'class_balance.csv', 'config.json', 'l2_search.csv', 'metrics.csv', 'per_class.csv']
